# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franciskendrick/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Action Rule (Snippet & Content Optimization Baseline):**

`AMONG` articles with at least $500$ impressions over $[t-30, t-1]$, not published within the last $60$ days, and not modified during a $30$-day cooldown period, `LOOK AT` their position-adjusted $\text{CTR}$, GA4 engagement rate, and impression decay. `IF` performance metrics fall significantly below position-peer benchmarks, `THEN` flag the article for human review with a specific reason code and prioritised action score.

In [4]:
import duckdb
from google.colab import userdata

# Initialize DuckDB & Authenticate via Secret Manager
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

WAREHOUSE_URI = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH_PATH = f"{WAREHOUSE_URI}/fact_content_daily_performance/month=2026-03/*.parquet"

query_signal_audit = f"""
SELECT
    content_hash_id,

    -- Corrected Column Binding & Weighted Position Aggregation
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,

    -- Impression-Weighted Average Position (Statistically Sound)
    ROUND(
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0), 2
    ) AS weighted_avg_position,

    -- Unweighted Daily Average Position (For Comparison Only)
    ROUND(AVG(gsc_avg_position), 2) AS unweighted_avg_position,

    -- Overall Click-Through Rate
    ROUND(
        100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2
    ) AS ctr_pct

FROM read_parquet('{FACT_MARCH_PATH}')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
  AND gsc_data_available IS TRUE
GROUP BY content_hash_id
HAVING SUM(gsc_impressions) > 100
ORDER BY total_impressions DESC
LIMIT 10;
"""

df_signal = con.execute(query_signal_audit).df()
print("--- SIGNAL CHECK 1: CORRECTED POSITION AGGREGATION ---")
print(df_signal.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SIGNAL CHECK 1: CORRECTED POSITION AGGREGATION ---
         content_hash_id  total_impressions  total_clicks  weighted_avg_position  unweighted_avg_position  ctr_pct
content_eadb33b5df496f4a           617124.0        5668.0                   2.33                     2.38     0.92
content_ec2e0346994fb5a5           245276.0        1480.0                   2.76                     2.85     0.60
content_e8a52cf3d5988c07           244931.0         669.0                  15.17                    15.01     0.27
content_0e03de7680314cd5           221310.0         720.0                   2.51                     2.68     0.33
content_44f34c0a90047651           212404.0          24.0                   0.67                     7.35     0.01
content_7172a7fad43f0998           205867.0         862.0                   3.30                     3.37     0.42
content_e7b5dd4dff461ad2           205045.0        2446.0                   4.45                     4.54     1.19
content_8d7d99f109e19aa2 

## 2. Build the ranked queue (writes the CSV)

**Score Formulation & Decision Logic:**

To order the actionable queue, we define a explicit scalar metric: Impression Opportunity at Risk. Rather than ranking by raw error or unweighted metrics, the action score directly measures expected traffic potential bounded by historical demand

$$\text{Action Score} = \text{Impressions}_{[t-30, t-1]} \times \max\left(0, \frac{\text{Peer Median CTR} - \text{Observed CTR}}{\text{Peer Median CTR}}\right)$$

This framing ensures that high-volume opportunities at Page 1 positions receive priority over minor positional shifts on low-traffic long-tail keywords.

In [11]:
import os

# Ensure output directory exists
os.makedirs("../outputs", exist_ok=True)
csv_output_path = "../outputs/baseline_action_score.csv"

WAREHOUSE_URI = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT_PATH = f"{WAREHOUSE_URI}/dim_content.parquet"

query_baseline_queue = f"""
SELECT
    f.report_date,
    f.client_hash_id,
    f.content_hash_id,
    c.content_created_date,
    c.word_count,
    c.search_volume,
    c.competition,
    c.cpc,
    -- Dynamically calculate content age at decision moment t
    DATEDIFF('day', c.content_created_date, f.report_date) AS content_age_days,
    -- Dynamically calculate freshness gap at decision moment t
    DATEDIFF('day', COALESCE(c.last_optimized_date, c.content_created_date), f.report_date) AS days_since_last_update,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position
FROM read_parquet('{FACT_MARCH_PATH}') f
LEFT JOIN read_parquet('{DIM_CONTENT_PATH}') c
    ON f.content_hash_id = c.content_hash_id
WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-31'
  AND f.gsc_data_available IS TRUE
  AND f.ga4_data_available IS TRUE
  AND c.content_created_date IS NOT NULL;
"""

df_queue = con.sql(query_baseline_queue).df()

# Save queue to CSV
df_queue.to_csv(csv_output_path, index=False)

print(f"=== BASELINE QUEUE GENERATED ===")
print(f"Rows Written to CSV : {len(df_queue):,}")
print(f"Output File Path    : {csv_output_path}")
print("\nTop 5 Queue Preview:")
print(df_queue[['report_date', 'content_hash_id', 'content_created_date', 'content_age_days', 'days_since_last_update']].head(5).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== BASELINE QUEUE GENERATED ===
Rows Written to CSV : 364,347
Output File Path    : ../outputs/baseline_action_score.csv

Top 5 Queue Preview:
report_date          content_hash_id content_created_date  content_age_days  days_since_last_update
 2026-03-01 content_810cf06597918291           2025-03-28               338                     -89
 2026-03-01 content_eb0aeedbcfaf2712           2025-03-28               338                     -89
 2026-03-01 content_b813c73d7000b3b1           2025-03-28               338                     -89
 2026-03-01 content_651b8ba180f9beff           2025-03-28               338                     -89
 2026-03-01 content_1f39e904c7351258           2025-03-28               338                     -89


## 3. Top-20 review

**Analytical Evaluation of Top-20 Priority Queue**

Relying purely on automated priority scores or heuristic ranking rules introduces systematic failure modes into production queues. When evaluating top-ranked pages at decision moment $t$ (using historical lookback features over $[t-90, t-1]$), a senior engineer must audit candidates against four structural false-positive patterns:
1. Low-Volume Volatility Traps: Pages with minimal baseline traffic (e.g., drop from 4 clicks to 2 clicks over 90 days) exhibit a $-50\%$ trend drop (trend_direction = 'down'). Ranking these at the top wastes human editorial capacity on statistically insignificant noise. Hard minimum-volume thresholds (e.g., $impressions_{90d} \ge 500$) are strictly required.

2. Keyword Cannibalization & Intent Consolidation: A page experiencing visibility loss may not be suffering from content decay. If a newer sibling page under the same client_hash_id or keyword_hash_id absorbs search intent, refreshing the declining URL creates internal competition. The true remediation is canonicalization or redirection, not content expansion.

3. SERP Layout & Zero-Click Shifts: Pages where impressions and average rank remain stable ($position \le 5$) while CTR plummets often suffer from external SERP feature additions (e.g., Google AI Overviews or featured snippets absorbing user intent). Content edits cannot recover zero-click search loss.

4. Tracking Disruption / Integration Lags: Discontinuities in tracking telemetry (e.g., gsc_data_available or ga4_data_available toggling to FALSE during the lookback interval) create synthetic traffic drops that appear as organic performance decay.

In [14]:
import pandas as pd
import numpy as np

# Load starter dataset
data_path = "https://raw.githubusercontent.com/franciskendrick/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Compute normalized baseline scoring components
df['norm_impressions'] = np.log1p(df['impressions_90d'].clip(lower=0)) / np.log1p(df['impressions_90d'].max())
df['norm_freshness'] = (df['content_age_days'].clip(upper=730) / 730.0)
df['norm_position_opp'] = df['avg_position'].apply(lambda pos: 1.0 if 1 <= pos <= 20 else 0.0)
df['norm_word_count_gap'] = df['word_count'].apply(lambda w: 1.0 if 0 < w < 1200 else 0.0)

df['baseline_refresh_score'] = (
    0.40 * df['norm_impressions'] +
    0.30 * df['norm_freshness'] +
    0.25 * df['norm_position_opp'] +
    0.05 * df['norm_word_count_gap']
)

# Extract Top-20 Priority Queue
top_20 = df.sort_values(by='baseline_refresh_score', ascending=False).head(20).copy()

# Audit Taxonomy for Structural Failure Modes in Top 20
def audit_false_positive_risk(row):
    risks = []
    # 1. SERP Feature / AI Overview Loss
    if row['avg_position'] <= 5 and row['ctr'] < 1.0 and row['impressions_90d'] > 1000:
        risks.append('SERP_AI_FEATURE_LOSS')
    # 2. High Impression / Zero Session Intent Mismatch
    if row['sessions_90d'] < 10 and row['impressions_90d'] > 500:
        risks.append('ZERO_CLICK_INTENT_MISMATCH')
    # 3. Striking Distance Volatility
    if 11 <= row['avg_position'] <= 20:
        risks.append('STRIKING_DISTANCE_VOLATILITY')
    # 4. Thin Content Noise
    if 0 < row['word_count'] < 500:
        risks.append('THIN_PAGE_NOISE')

    return '|'.join(risks) if risks else 'VALID_CANDIDATE'

top_20['fp_risk_taxonomy'] = top_20.apply(audit_false_positive_risk, axis=1)

# Display Top-20 Structural Failure Mode Evaluation
output_cols = [
    'content_id', 'impressions_90d', 'sessions_90d',
    'avg_position', 'ctr', 'trend_direction',
    'baseline_refresh_score', 'fp_risk_taxonomy'
]

print("=== TOP-20 PRIORITY QUEUE STRUCTURAL FAILURE MODE AUDIT ===")
print(top_20[output_cols].round(4).to_string(index=False))

# Breakdown of Systematic Vulnerabilities
fp_breakdown = top_20['fp_risk_taxonomy'].value_counts()
print("\n=== FALSE POSITIVE RISK DISTRIBUTION IN TOP 20 ===")
print(fp_breakdown.to_string())

=== TOP-20 PRIORITY QUEUE STRUCTURAL FAILURE MODE AUDIT ===
          content_id  impressions_90d  sessions_90d  avg_position  ctr trend_direction  baseline_refresh_score     fp_risk_taxonomy
content_5fe46e04994d           517715           520           4.2 0.14            down                  0.8707 SERP_AI_FEATURE_LOSS
content_1a9e894be2e2           416180          1140           4.0 0.23            down                  0.8414 SERP_AI_FEATURE_LOSS
content_aaef01a50def           517109          1527           5.4 0.25          stable                  0.8328      VALID_CANDIDATE
content_8c19996aa890           509252           571           2.5 0.15            down                  0.8324 SERP_AI_FEATURE_LOSS
content_4c36c775b818           463103          1114           2.3 0.41            down                  0.8295 SERP_AI_FEATURE_LOSS
content_9b934e3e7101           106384           244           7.1 0.40          stable                  0.8226      VALID_CANDIDATE
content_db5989a7

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.